# Bank Statement Automation — Live Demo

**Gmail → Drive → Parse → Analyse → Visualise → Cross-check → Excel**

Click **Runtime → Run all** to see the full pipeline execute on synthetic bank statements.

---

In [ ]:
# Clone the repo and install dependencies
!git clone https://github.com/RohitS2497/bank-statement-automation.git 2>/dev/null
%cd bank-statement-automation
!pip -q install pdfplumber pypdf reportlab openpyxl python-dateutil

In [ ]:
# Stage 0: Generate synthetic bank statement PDFs (simulates what Gmail+Drive delivers)
!python gen_data.py

In [ ]:
# Run the full 6-stage pipeline
import statement_pipeline as sp
sp.run()

In [ ]:
# Display the charts inline
from IPython.display import Image, display
import os

for chart in sorted(os.listdir('outputs/charts')):
    print(f'\n--- {chart} ---')
    display(Image(filename=f'outputs/charts/{chart}', width=600))

In [ ]:
# Show the validation flags (cross-check catches deliberate anomalies)
import pandas as pd
df = pd.read_excel('outputs/Bank_Statement_Analysis.xlsx', sheet_name='Validation Flags')
df

In [ ]:
# Show file status (edge case handling)
fs = pd.read_excel('outputs/Bank_Statement_Analysis.xlsx', sheet_name='File Status')
fs

In [ ]:
# Transaction summary
txns = pd.read_excel('outputs/Bank_Statement_Analysis.xlsx', sheet_name='Transactions')
print(f"Total transactions: {len(txns)}")
print(f"Total income: INR {txns['Credit'].sum():,.2f}")
print(f"Total expense: INR {txns['Debit'].sum():,.2f}")
print(f"\nCategory breakdown:")
pd.read_excel('outputs/Bank_Statement_Analysis.xlsx', sheet_name='Category Summary')

---
## Output files generated

| File | Purpose |
|------|--------|
| `outputs/Bank_Statement_Analysis.xlsx` | Multi-sheet workbook with Dashboard, Transactions, Summaries, Validation Flags, File Status |
| `outputs/transactions_clean.csv` | Flat CSV for downstream use |
| `outputs/charts/*.png` | Spend by category, Income vs Expense, Balance trend |
| `outputs/run_log.txt` | Full audit trail of pipeline execution |

---

## Architecture

| Stage | Tool | Runs where |
|-------|------|------------|
| **Fetch + Store** | `FetchStatements.gs` (Apps Script) | Inside Google account, on hourly trigger |
| **Parse → Analyse → Visualise → Cross-check → Export** | `statement_pipeline.py` | Google Colab (this notebook) or local |

The Apps Script has native Gmail+Drive access (no OAuth JSON needed). Python handles the heavy lifting.

### To connect to a real Gmail account:
1. Paste `FetchStatements.gs` into script.google.com
2. Run once to grant permissions
3. Mount Drive in this notebook and point `sp.INBOX` at the Drive folder

See `README.md` for full setup instructions.